# Finding Wildfires from AEF 64-band Embeddings with Rasteret

This notebook shows a compact remote-sensing workflow built on three pieces:

1. **Rasteret** loads a published remote AEF collection and reads pixels from it.
2. **TorchGeo** lays out chip queries over a study area.
3. **LanceDB** scores pooled chip embeddings by cosine similarity.

The goal is not to train a classifier. The goal is to test whether AEF's 64-band
embeddings already separate fire-like regions well enough for a simple retrieval
workflow.

We use Rasteret in three places:

1. `rasteret.load("aef/v1-annual")` to reopen the published AEF collection from Source Cooperative.
2. `collection.sample_points()` to read 64-band embedding vectors at known fire and background points.
3. `collection.to_torchgeo_dataset()` to serve gridSample chips and later put them into LanceDB

**On an AWS US-west-2 4 core VM, this tutorial should finish running in about 5 minutes.**
**And repeated runs use cached vector data and LanceDB tables, won't query COGs again, should run in 1 minute.**

`This notebook is not meant to show the best way to use AEF, it's to show how Rasteret helps in reducing LinesOfCode, increase speed of iteration, and easier interop with arrow native tools`

---

### Libraries

| Library | What it does here |
|---------|-------------------|
| [**Rasteret**](https://github.com/terrafloww/rasteret) | Opens the published AEF collection, filters it, samples points, and serves chips into TorchGeo. |
| [**TorchGeo**](https://github.com/microsoft/torchgeo) | Defines chip queries over a geographic study area. |
| [**LanceDB**](https://lancedb.com/) | Stores pooled chip vectors and scores them with cosine distance. |
| [**Lonboard**](https://developmentseed.org/lonboard/latest/) | Renders Arrow / GeoArrow geometry directly in the notebook. |
| [**MTBS**](https://www.mtbs.gov/) | Provides wildfire perimeter ground truth for evaluation. |


## Setup

Install the small set of libraries this notebook needs for remote AEF access,
TorchGeo sampling, vector search, and interactive maps.

```bash
uv add rasteret lancedb torchgeo lonboard geoarrow-pyarrow matplotlib
```

If you are running this after opening the repo in editable mode, `rasteret` itself
can already come from the local checkout. The rest are notebook-only dependencies.


In [ ]:
import datetime
import os
import time
from pathlib import Path

import geopandas as gpd
import numpy as np

try:
    import lancedb
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "lancedb is required for this notebook. Install it first: uv add lancedb"
    ) from exc
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
from lonboard import Map, PolygonLayer, ScatterplotLayer
from lonboard.basemap import CartoStyle, MaplibreBasemap
from lonboard.colormap import apply_categorical_cmap
from shapely import box
from torch.utils.data import DataLoader
from torchgeo.datasets.utils import stack_samples
from torchgeo.samplers import GridGeoSampler

import rasteret

os.environ.setdefault("AWS_NO_SIGN_REQUEST", "YES")

NOTEBOOK_T0 = time.perf_counter()

DARK_BASEMAP = MaplibreBasemap(style=CartoStyle.DarkMatter)

## Configuration

All the knobs in one place. The seed fire is a 2020 California wildfire; the scan
area is centred on 2022 year fires in large area near Lousiana.

Using different years tests whether the AEF embeddings generalise across time.

The `SCAN_CACHE` path includes the grid parameters in the filename, so changing
chip size, stride, or band count automatically creates a new cache file.

In [ ]:
# Data sources
MTBS_URL = "https://data.source.coop/cboettig/fire/usgs-mtbs.parquet"
MTBS_CACHE = Path(".cache/mtbs/usgs-mtbs.parquet")
COLLECTION_URI = "aef/v1-annual"

# Seed fire: used only to define the prototype direction in embedding space.
SEED_EVENT_ID = "CA3726212222320200816"  # CZU Aug Lightning Complex, 2020
SEED_YEAR = 2020

# Scan area: separate region/year where we test whether the prototype transfers.
SCAN_EVENT_ID = "LA3103109293720221202"  # known 2022 Louisiana fire
SCAN_BUFFER_DEG = 0.30
SCAN_YEAR = 2022

# AEF exposes 64 embedding bands; we read the full vector for this workflow.
BANDS = [f"A{i:02d}" for i in range(64)]

# Chip grid for the TorchGeo scan.
CHIP_SIZE = 1024
CHIP_STRIDE = 1024
BATCH_SIZE = 4
MAX_CHIPS = None

# Prototype construction and retrieval cutoff.
GEM_P = 3.0
N_FIRE_PTS = 24
N_BACKGROUND = 64
PREDICT_TOP_PCT = 0.10

# Vector-search artifacts and exported map outputs live together.
LANCEDB_DIR = Path(".cache/aef_fire_lancedb_v3")
LANCEDB_TABLE = "aef_fire_chips"
MAP_HTML = LANCEDB_DIR / "aef_fire_webmap.html"

# Cache pooled chip vectors, not raw 64-band chip arrays.
SCAN_CACHE = (
    LANCEDB_DIR
    / f"scan_rows_y{SCAN_YEAR}_cs{CHIP_SIZE}_st{CHIP_STRIDE}_n{MAX_CHIPS or 'all'}_b{len(BANDS)}.parquet"
)
REUSE_SCAN_CACHE = True

## Helpers

Utility functions used throughout the notebook:

- **`dequantize_aef`** — AEF stores embeddings as int8 to save bandwidth. This
  converts them back to float32.
- **`gem_pool`** — Generalized Mean pooling. Compresses a `[C, H, W]` image chip
  into one unit-length vector. More info in this [paper](https://arxiv.org/html/2603.02080v1)
- **`random_points_in_polygon`** — Generates random points inside a polygon using
  rejection sampling.
- **`build_prototype`** — Computes the "fire direction" in embedding space:
  mean(fire pixels) − mean(background pixels), normalised to unit length.
- **`rasteret_sample_and_pivot`** — Reads pixel values at specific points using
  Rasteret's `sample_points()`, which fetches all 64 bands concurrently. Returns
  the results pivoted to wide table (one row per point, one column per band).

In [ ]:
AEF_NODATA = -128


def dequantize_aef(values):
    """Convert AEF int8 values back to float32. Nodata (-128) becomes NaN."""
    # AEF stores quantized signed int8 embeddings; undo that transform for pooling.
    raw = values.astype(np.float32, copy=False)
    out = ((raw / 127.5) ** 2) * np.sign(raw)
    out[raw == AEF_NODATA] = np.nan
    return out


def gem_pool(cube, p=3.0):
    """Pool a [C, H, W] image chip into one unit-length vector."""
    deq = dequantize_aef(cube)
    # Flatten spatial positions so pooling works on one [N_pixels, C] matrix.
    flat = np.moveaxis(deq, 0, -1).reshape(-1, deq.shape[0])
    valid = np.isfinite(flat).all(axis=1)
    if not valid.any():
        return np.full(deq.shape[0], np.nan, dtype=np.float32), 0
    flat = flat[valid]
    pooled = np.sign(flat).mean(axis=0) * (
        np.mean(np.abs(flat) ** p, axis=0) ** (1.0 / p)
    )
    pooled = pooled.astype(np.float32)
    norm = np.linalg.norm(pooled)
    if np.isfinite(norm) and norm > 0:
        pooled /= norm
    return pooled, int(flat.shape[0])


def gem_pool_batch(cubes, p=3.0):
    """Pool a batch of [B, C, H, W] chips into [B, C] vectors."""
    deq = dequantize_aef(cubes)
    # Move channels last so batch pooling can work with one [B, N_pixels, C] array.
    flat = np.moveaxis(deq, 1, -1).reshape(deq.shape[0], -1, deq.shape[1])
    valid = np.isfinite(flat).all(axis=2)
    valid_counts = valid.sum(axis=1).astype(np.int32)

    # Zero-fill invalid pixels so the reductions stay vectorized across the batch.
    safe = np.where(valid[..., None], flat, 0.0)
    denom = np.maximum(valid_counts, 1).astype(np.float32)[:, None]

    sign_mean = np.sign(safe).sum(axis=1) / denom
    abs_pow_mean = (np.abs(safe) ** p).sum(axis=1) / denom
    pooled = (sign_mean * np.power(abs_pow_mean, 1.0 / p)).astype(np.float32)
    pooled[valid_counts == 0] = np.nan

    norms = np.linalg.norm(pooled, axis=1, keepdims=True)
    good = np.isfinite(norms[:, 0]) & (norms[:, 0] > 0)
    pooled[good] /= norms[good]

    return pooled, valid_counts


def random_points_in_polygon(poly, n, label, seed=42):
    """Sample n random points inside a polygon using rejection sampling."""
    # Deterministic sampling makes repeated notebook runs easier to compare.
    rng = np.random.default_rng(seed)
    minx, miny, maxx, maxy = poly.bounds
    pts = []
    while len(pts) < n:
        x, y = rng.uniform(minx, maxx), rng.uniform(miny, maxy)
        if poly.contains(gpd.points_from_xy([x], [y])[0]):
            pts.append((x, y))
    lons, lats = zip(*pts)
    return gpd.GeoDataFrame(
        {"point_id": range(n), "label": label, "lon": list(lons), "lat": list(lats)},
        geometry=gpd.points_from_xy(lons, lats),
        crs="EPSG:4326",
    )


def build_prototype(anchors_wide, bands):
    """Compute the fire prototype: mean(fire) - mean(background), normalised."""
    fire_mask = pc.equal(anchors_wide.column("label"), "fire")
    bg_mask = pc.equal(anchors_wide.column("label"), "background")
    # Build two dense [N_points, C] matrices before taking the mean difference.
    fire = dequantize_aef(
        np.column_stack(
            [anchors_wide.filter(fire_mask).column(b).to_numpy() for b in bands]
        )
    )
    bg = dequantize_aef(
        np.column_stack(
            [anchors_wide.filter(bg_mask).column(b).to_numpy() for b in bands]
        )
    )
    diff = (np.nanmean(fire, axis=0) - np.nanmean(bg, axis=0)).astype(np.float32)
    norm = float(np.linalg.norm(diff))
    return diff if norm == 0 else diff / norm


def rasteret_sample_and_pivot(collection, points, bands):
    """Read pixel values at given points using Rasteret, return wide-format Arrow table.

    Rasteret's sample_points() fetches all 64 bands concurrently from cloud COGs.
    We pivot the long-format result (one row per point-band) to wide format
    (one row per point, one column per band)."""
    samples = collection.sample_points(
        points=points, geometry_column="geometry", bands=bands
    )
    wide = (
        samples.to_pandas()
        .pivot(index="point_index", columns="band", values="value")
        .reset_index()
    )
    wide.columns.name = None
    return pa.Table.from_pandas(wide)

## Fire perimeters (ground truth)

[MTBS](https://www.mtbs.gov/) (Monitoring Trends in Burn Severity) maps every US
fire larger than 1,000 acres going back to 1984. The full dataset is published as
GeoParquet on Source Cooperative. We download it once and cache it locally — subsequent
runs read straight from disk.

In [ ]:
def ensure_mtbs_cache():
    # Reuse a valid local copy when present; otherwise refresh it atomically.
    if MTBS_CACHE.exists() and MTBS_CACHE.stat().st_size > 0:
        try:
            pq.ParquetFile(MTBS_CACHE)
            return MTBS_CACHE
        except Exception:
            MTBS_CACHE.unlink()
    MTBS_CACHE.parent.mkdir(parents=True, exist_ok=True)
    from urllib.request import Request, urlopen

    tmp = MTBS_CACHE.with_suffix(".tmp")
    tmp.unlink(missing_ok=True)
    print("Downloading MTBS parquet...")
    request = Request(MTBS_URL, headers={"User-Agent": "Mozilla/5.0"})
    with urlopen(request) as response:
        tmp.write_bytes(response.read())
    tmp.replace(MTBS_CACHE)
    return MTBS_CACHE


mtbs_path = ensure_mtbs_cache()
print(f"MTBS cached at {mtbs_path}")

## Step 1: Load the AEF collection with Rasteret

Good news: the AEF Rasteret collection is already published for **2018–2024**
on **Source Cooperative** and **Hugging Face**.

This notebook uses `rasteret.load("aef/v1-annual")` to reopen the
published Collection directly. The narrow `index.parquet` remains in front
for pushdown, so you do not need to stage or ingest the source data by hand.


In [ ]:
rasteret.set_options(progress=False)
collection = rasteret.load(COLLECTION_URI, name="aef-fire-lancedb-torchgeo")

collection.describe()

## Step 2: Pick a seed fire and sample anchor points

We need to tell the algorithm what "fire" looks like in AEF's 64-dimensional embedding
space. The 2020 CZU August Lightning Complex in California is a well-documented
wildfire — we'll use its MTBS perimeter.

We scatter random points inside the fire polygon. In Step 3, Rasteret will read all
64 AEF bands at each of these points. Background points (for contrast) come from
the scan area later in Step 5.

In [ ]:
seed_gdf = gpd.read_parquet(
    MTBS_CACHE, filters=[("Event_ID", "=", SEED_EVENT_ID)]
).to_crs("EPSG:4326")

# Keep the seed geometry in EPSG:4326 because Rasteret subset/sample calls below use lon/lat bounds.
seed_poly = seed_gdf.geometry.iloc[0]
seed_name = seed_gdf["Incid_Name"].iloc[0]
seed_date = seed_gdf["Ig_Date"].iloc[0]
seed_bbox = tuple(seed_poly.bounds)

print(f"Seed fire: {seed_name} ({seed_date})")

fire_pts = random_points_in_polygon(seed_poly, N_FIRE_PTS, "fire", seed=42)
anchor_gdf = fire_pts.copy()
anchor_gdf["point_id"] = range(len(anchor_gdf))

print(f"{len(anchor_gdf)} fire anchor points")
anchor_gdf[["point_id", "label", "lon", "lat"]].head()

Red dots = fire anchor points inside the burn perimeter.

This preview is here for a simple sanity check before we read any AEF pixels:

- the MTBS seed perimeter is the fire we intend to learn from
- the sampled anchor points really fall inside that perimeter
- the geometry is in the CRS we expect for later sampling

To pass GeoDataFrames to lonboard, we use `gdf.to_arrow()` which encodes geometry
as [GeoArrow](https://geoarrow.org/) WKB.


In [ ]:
# Lonboard reads Arrow/GeoArrow tables directly, so we can stay out of file export paths here.
seed_arrow = seed_gdf.to_crs("EPSG:4326")[["Incid_Name", "geometry"]].to_arrow()
anchor_arrow = anchor_gdf.to_crs("EPSG:4326").to_arrow()

Map(
    layers=[
        PolygonLayer(
            seed_arrow,
            opacity=0.25,
            get_fill_color=[255, 255, 255, 60],
            get_line_color=[0, 191, 255, 200],
            stroked=True,
            get_line_width=2,
            line_width_units="pixels",
        ),
        ScatterplotLayer(
            anchor_arrow, get_fill_color=[239, 68, 68], radius_min_pixels=5
        ),
    ],
    basemap=DARK_BASEMAP,
)

## Step 3: Read AEF pixel values with Rasteret

> **Rasteret call 1 of 3** — `collection.sample_points()`: read all 64 AEF bands
> at the fire anchor points.

`collection.subset(bbox, date_range)` narrows the collection lazily to records that
match the fire area and year. No pixels are read yet.

Then `sample_points()` does the actual data-plane work. Rasteret resolves which remote
rasters intersect the requested points and reads the requested band values from the
published collection.

The result comes back as a PyArrow table in long format: one row per point-band pair.
We pivot it to wide format so each point has one 64-dimensional embedding vector.


In [ ]:
seed_sub = collection.subset(
    bbox=seed_bbox,
    date_range=(f"{SEED_YEAR}-01-01", f"{SEED_YEAR}-12-31"),
)

t0 = time.perf_counter()

anchors_wide = rasteret_sample_and_pivot(seed_sub, anchor_gdf, BANDS)

print(
    f"Sampled {anchors_wide.num_rows} points × {len(BANDS)} bands in {time.perf_counter() - t0:.2f}s"
)

Merge the labels back so every row is self-contained.

`sample_points()` returns the band values we need, but the original point labels
(`fire`, longitude, latitude) still live in the input GeoDataFrame. Joining them
back makes the sampled Arrow table easier to inspect and reuse in later steps.


In [ ]:
# Build a tiny Arrow lookup table so the merge stays in Arrow instead of bouncing through pandas.
anchor_lookup = pa.table(
    {
        "point_index": pa.array(anchor_gdf["point_id"].to_numpy()),
        "label": pa.array(anchor_gdf["label"].to_numpy()),
        "lon": pa.array(anchor_gdf["lon"].to_numpy()),
        "lat": pa.array(anchor_gdf["lat"].to_numpy()),
    }
)

anchors_wide = anchors_wide.join(anchor_lookup, keys="point_index")

print(f"{anchors_wide.num_rows} points, showing first 6 of {len(BANDS)} bands:")
anchors_wide.select(["point_index", "label"] + BANDS[:6]).slice(0, 5).to_pandas()

## Step 4: Tile the study area with TorchGeo + Rasteret

> **Rasteret call 2 of 3** — `collection.to_torchgeo_dataset()`: wrap the collection
> as a TorchGeo `GeoDataset` so a standard `DataLoader` can read 64-band chips.

Now we switch from point reads to area reads.

`to_torchgeo_dataset()` keeps Rasteret in charge of pixel access while exposing a
standard TorchGeo dataset interface. TorchGeo decides *which chip bounds to request*, and Rasteret handles *reading the
chip pixels* for those bounds.

`GridGeoSampler` lays out a regular chip grid over the study area. The notebook uses
`num_workers=0` for a stable interactive run; in training scripts you would tune
DataLoader workers normally.

Each `[64, H, W]` chip is pooled down to one 64-d vector with GeM pooling. We cache
these Gem pooled vectors in Parquet so later notebook iterations can focus on scoring and
visualization instead of re-reading pixels.


In [ ]:
# Read the scan-year fire perimeter, then expand it to a study area.
scan_gdf = gpd.read_parquet(
    MTBS_CACHE,
    filters=[("Event_ID", "=", SCAN_EVENT_ID)],
    columns=["Incid_Name", "Ig_Date", "geometry"],
).to_crs("EPSG:4326")

scan_name = scan_gdf["Incid_Name"].iloc[0]
scan_date = scan_gdf["Ig_Date"].iloc[0]

study_roi = scan_gdf.geometry.iloc[0].buffer(SCAN_BUFFER_DEG)
scan_bbox = tuple(study_roi.bounds)

# Keep the collection lazy here; the actual chip reads happen via TorchGeo below.
scan_sub = collection.subset(
    bbox=scan_bbox,
    date_range=(f"{SCAN_YEAR}-01-01", f"{SCAN_YEAR}-12-31"),
)

# TorchGeo owns the spatial query interface; Rasteret still owns pixel access.
tg_ds = scan_sub.to_torchgeo_dataset(
    bands=BANDS,
    geometries=study_roi.wkb,
    chip_size=CHIP_SIZE,
    time_series=False,
)
scan_crs = int(tg_ds.epsg)

print(f"Scan fire: {scan_name} ({scan_date})")
print(f"Dataset CRS: EPSG:{scan_crs}")

In [ ]:
vec_size = len(BANDS)

# Reuse pooled chip vectors when the cache already contains a valid geometry-bearing table.
cache_valid = REUSE_SCAN_CACHE and SCAN_CACHE.exists()
if cache_valid:
    chips_table = pq.read_table(SCAN_CACHE)
    if "geometry" not in chips_table.column_names or chips_table.num_rows == 0:
        cache_valid = False

if cache_valid:
    # Warm path: skip remote chip reads and jump straight to scoring/evaluation.
    print(f"Loaded {chips_table.num_rows} cached chip vectors from {SCAN_CACHE.name}")
else:
    sampler = GridGeoSampler(tg_ds, size=CHIP_SIZE, stride=CHIP_STRIDE)

    # The sampler gives us the chip footprints up front; pixel I/O starts with the DataLoader.
    chip_bounds = [(x.start, y.start, x.stop, y.stop) for x, y, _ in sampler]
    print(f"Scanning {len(chip_bounds)} chips × {vec_size} bands via Rasteret...")

    loader = DataLoader(
        tg_ds,
        sampler=sampler,
        batch_size=BATCH_SIZE,
        num_workers=0,
        collate_fn=stack_samples,
    )
    valid_bounds, valid_vecs = [], []
    seen = 0

    t0 = time.perf_counter()
    for batch_idx, batch in enumerate(loader):
        # TorchGeo batches standard image tensors; pool the full batch before touching Python lists.
        imgs = batch["image"].cpu().numpy()
        vecs, valid_counts = gem_pool_batch(imgs, p=GEM_P)

        start = batch_idx * BATCH_SIZE
        for offset, (vec, count) in enumerate(zip(vecs, valid_counts, strict=False)):
            if MAX_CHIPS is not None and seen >= MAX_CHIPS:
                break
            bounds = chip_bounds[start + offset]
            seen += 1
            if count == 0 or not np.isfinite(vec).all():
                continue
            valid_bounds.append(bounds)
            valid_vecs.append(vec)
        if MAX_CHIPS is not None and seen >= MAX_CHIPS:
            break

    print(f"Read and pooled {seen} 64-band chips in {time.perf_counter() - t0:.1f}s")

    # Persist pooled vectors together with chip footprints so later runs can stay metadata-only.
    SCAN_CACHE.parent.mkdir(parents=True, exist_ok=True)
    chips_gdf = gpd.GeoDataFrame(
        {
            "chip_id": np.arange(len(valid_bounds), dtype=np.int32),
            "vector": valid_vecs,
        },
        geometry=[box(*b) for b in valid_bounds],
        crs=f"EPSG:{scan_crs}",
    ).to_crs("EPSG:4326")

    # GeoPandas owns the friendly table assembly; Arrow is only the Parquet/LanceDB handoff.
    chips_table = pa.table(chips_gdf.to_arrow())

    pq.write_table(chips_table, SCAN_CACHE)
    print(f"Cached to {SCAN_CACHE.name}")

## Step 5: Build the fire prototype

> **Rasteret call 3 of 3** — `collection.sample_points()` again: read all 64 AEF
> bands at background points in the scan area.

We need background points to contrast against fire. A good background sample should
represent normal land in the study area without landing on recent burn scars.

We compute `grid_extent` from the actual chip footprints, exclude MTBS fire perimeters
from 2020 through the scan year, and sample background points from the remaining area.

Rasteret reads the 64-band AEF vectors at those background points just as it did for
our fire anchor points. The prototype vector is then:

`mean(fire embeddings) - mean(background embeddings)`

normalized to unit length.


In [ ]:
# Load same-year MTBS perimeters for evaluation against the retrieved chips.
fires_gdf = gpd.read_parquet(
    MTBS_CACHE,
    filters=[
        ("Ig_Date", ">=", datetime.date(SCAN_YEAR, 1, 1)),
        ("Ig_Date", "<=", datetime.date(SCAN_YEAR, 12, 31)),
    ],
    columns=["Event_ID", "Incid_Name", "geometry"],
).to_crs("EPSG:4326")

# Use the realized chip footprints, not the requested buffer, as the evaluation extent.
chip_geoms = gpd.GeoDataFrame.from_arrow(chips_table.select(["geometry"])).to_crs(
    "EPSG:4326"
)

grid_extent = box(*chip_geoms.total_bounds)

fires_in_area = fires_gdf[fires_gdf.intersects(grid_extent)]
print(f"{len(fires_in_area)} MTBS fires in grid extent ({SCAN_YEAR})")

# Exclude recent burn scars so background points do not learn a fire-like baseline.
fires_all_years = gpd.read_parquet(
    MTBS_CACHE,
    filters=[
        ("Ig_Date", ">=", datetime.date(2020, 1, 1)),
        ("Ig_Date", "<=", datetime.date(SCAN_YEAR, 12, 31)),
    ],
    columns=["Event_ID", "geometry"],
).to_crs("EPSG:4326")

fires_to_exclude = fires_all_years[fires_all_years.intersects(grid_extent)]
print(f"{len(fires_to_exclude)} fires (2020–{SCAN_YEAR}) excluded from background")

# Union the exclusion perimeters once so the background mask stays simple.
fire_union = fires_to_exclude.union_all() if len(fires_to_exclude) > 0 else None

bg_area = grid_extent.difference(fire_union) if fire_union else grid_extent
bg_pts = random_points_in_polygon(bg_area, N_BACKGROUND, "background", seed=44)
print(f"Generated {len(bg_pts)} background points")

t0 = time.perf_counter()
background_wide = rasteret_sample_and_pivot(
    collection.subset(
        bbox=tuple(grid_extent.bounds),
        date_range=(f"{SCAN_YEAR}-01-01", f"{SCAN_YEAR}-12-31"),
    ),
    bg_pts,
    BANDS,
)

print(
    f"Sampled {background_wide.num_rows} background points × {len(BANDS)} bands in {time.perf_counter() - t0:.2f}s"
)

# Join the background labels back in so prototype construction sees both classes explicitly.
background_lookup = pa.table(
    {
        "point_index": pa.array(bg_pts["point_id"].to_numpy()),
        "label": pa.array(bg_pts["label"].to_numpy()),
        "lon": pa.array(bg_pts["lon"].to_numpy()),
        "lat": pa.array(bg_pts["lat"].to_numpy()),
    }
)
background_wide = background_wide.join(background_lookup, keys="point_index")

Combine the fire and background samples, then build one prototype vector.

This is the simplest possible retrieval setup: one vector that points toward fire
in embedding space, built from the difference between the mean fire embedding and
the mean background embedding.


In [ ]:
# Stack fire + background samples so the prototype is built from one self-contained Arrow table.
anchors_with_bg = pa.concat_tables(
    [anchors_wide, background_wide], promote_options="default"
)

# One prototype vector is enough for this notebook because we are doing nearest-neighbor retrieval, not training.
query_vec = build_prototype(anchors_with_bg, BANDS)

print(f"Prototype shape: {query_vec.shape}")
print(f"Prototype L2 norm: {np.linalg.norm(query_vec):.3f}")

## Step 6: Vector search with LanceDB

`chips_table` is already a PyArrow table with a vector column and a GeoArrow geometry
column, so LanceDB can ingest it directly.

This is the second interoperability point worth noticing: Rasteret handles remote pixel
reads, TorchGeo handles chip queries, and LanceDB handles vector search, but the data
shape stays simple enough to move between them without building a custom pipeline.


In [ ]:
LANCEDB_DIR.mkdir(parents=True, exist_ok=True)
db = lancedb.connect(str(LANCEDB_DIR))

# LanceDB can ingest the pooled Arrow table directly; no custom export step needed.
tbl = db.create_table(LANCEDB_TABLE, data=chips_table, mode="overwrite")

# Score every chip against the single fire prototype vector.
results = (
    tbl.search(query_vec.tolist())
    .metric("cosine")
    .limit(chips_table.num_rows)
    .to_arrow()
)

print(f"Scored all {results.num_rows} chips:")
results.select(["chip_id", "_distance"]).slice(0, 5).to_pandas()

Flag the top 10% lowest-distance chips as predicted fires.

This keeps the scoring rule intentionally simple for a tutorial notebook. We are not
training a classifier or fitting a calibrated threshold here; we are asking whether
chips nearest to the fire prototype in embedding space line up with known fire areas.


In [ ]:
distances = results.column("_distance").to_numpy()
threshold = float(np.percentile(distances, PREDICT_TOP_PCT * 100))

# Keep score + predicted as Arrow columns so the result table stays easy to hand off downstream.
rows_table = results.append_column(
    "score", pc.subtract(pa.scalar(1.0), results.column("_distance"))
).append_column(
    "predicted", pc.less_equal(results.column("_distance"), pa.scalar(threshold))
)

n_pred = pc.sum(rows_table.column("predicted")).as_py()
print(f"Predicted fire: {n_pred} / {rows_table.num_rows} chips")

rows_table.select(["chip_id", "_distance", "score", "predicted"]).slice(
    0, 10
).to_pandas()

## Step 7: How did we do?

We compare the predicted chips against MTBS fire perimeters with a spatial join.

This is a retrieval-style evaluation, not a calibrated fire detector. The threshold is
simply the top 10% lowest cosine distances, and each prediction is a full chip footprint.
So the precision and recall numbers here should be read as a quick sanity check on the
workflow, not as benchmark-quality fire-mapping metrics.


In [ ]:
# Convert back to GeoDataFrame only for the spatial join / evaluation step.
chips_gdf = gpd.GeoDataFrame.from_arrow(
    rows_table.select(["chip_id", "geometry", "predicted", "score"])
).to_crs("EPSG:4326")
fires_eval = fires_in_area.to_crs("EPSG:4326")

# Chip-level retrieval is counted as correct if the chip footprint intersects MTBS fire geometry.
joined = gpd.sjoin(
    chips_gdf, fires_eval[["geometry"]], how="left", predicate="intersects"
)
overlap_ids = set(joined.loc[joined["index_right"].notna(), "chip_id"].tolist())
chips_gdf["overlaps_fire"] = chips_gdf["chip_id"].isin(overlap_ids)

tp = int((chips_gdf["predicted"] & chips_gdf["overlaps_fire"]).sum())
fp = int((chips_gdf["predicted"] & ~chips_gdf["overlaps_fire"]).sum())
fn = int((~chips_gdf["predicted"] & chips_gdf["overlaps_fire"]).sum())
tn = int((~chips_gdf["predicted"] & ~chips_gdf["overlaps_fire"]).sum())

print(f"TP={tp}  FP={fp}  FN={fn}  TN={tn}")
if tp + fp > 0:
    print(f"Precision: {tp/(tp+fp):.0%}")
if tp + fn > 0:
    print(f"Recall:    {tp/(tp+fn):.0%}")

## Results

This static figure is the quick readout of the retrieval run:

- red outlines = chips we flagged as fire-like
- orange outlines = chips that overlap a known MTBS fire perimeter
- cyan outlines = MTBS fire perimeters themselves

The point is not to produce a polished fire map. The point is to see, at a glance,
whether a simple AEF embedding retrieval workflow is already isolating the right parts
of the landscape.


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 8))
# Draw all chip footprints first, then overlay predictions and MTBS perimeters on top.
for g in chips_gdf.geometry:
    ax.plot(*g.exterior.xy, color="#9ca3af", linewidth=0.6)
for g in chips_gdf[chips_gdf["predicted"]].geometry:
    ax.plot(*g.exterior.xy, color="#ef4444", linewidth=1.2)
for g in chips_gdf[chips_gdf["overlaps_fire"]].geometry:
    ax.plot(*g.exterior.xy, color="#ff8c00", linewidth=1.6)
for g in fires_eval.geometry:
    for poly in getattr(g, "geoms", [g]):
        ax.plot(*poly.exterior.xy, color="deepskyblue", linewidth=0.9)
ax.set_title(f"AEF {SCAN_YEAR}: predicted (red) vs MTBS fire perimeters (cyan)")
ax.set_axis_off()

## Interactive map

Cyan = MTBS fire perimeters (ground truth), orange = true positive, red = false
positive, pale yellow = false negative, light grey = true negative.

The live lonboard widget is best in an interactive notebook session. This notebook also
exports a standalone HTML map so the result is still inspectable in static environments.


In [ ]:
# Collapse the boolean evaluation columns into one status label for mapping.
chips_gdf["status"] = np.select(
    [
        chips_gdf["predicted"] & chips_gdf["overlaps_fire"],
        chips_gdf["predicted"] & ~chips_gdf["overlaps_fire"],
        ~chips_gdf["predicted"] & chips_gdf["overlaps_fire"],
    ],
    ["TP", "FP", "FN"],
    default="TN",
)

chip_colors = apply_categorical_cmap(
    pa.array(chips_gdf["status"]),
    {
        "TP": [255, 160, 0],
        "FP": [239, 68, 68],
        "FN": [255, 220, 120],
        "TN": [220, 220, 220],
    },
)

fires_arrow = fires_eval[["Event_ID", "Incid_Name", "geometry"]].to_arrow()
chips_arrow = chips_gdf[["chip_id", "status", "geometry"]].to_arrow()

fires_layer = PolygonLayer(
    fires_arrow,
    get_fill_color=[0, 191, 255, 160],
    get_line_color=[0, 220, 255, 255],
    stroked=True,
    get_line_width=2,
    line_width_units="pixels",
)

chips_layer = PolygonLayer(
    chips_arrow,
    get_fill_color=chip_colors,
    opacity=0.4,
    get_line_color=[255, 255, 255, 60],
    stroked=True,
    get_line_width=1,
    line_width_units="pixels",
)

print(f"Total time: {time.perf_counter() - NOTEBOOK_T0:.1f}s")

m = Map(layers=[fires_layer, chips_layer], basemap=DARK_BASEMAP)
try:
    m.to_html(MAP_HTML)
    print(f"Saved interactive map to {MAP_HTML}")
except Exception as exc:
    print(f"Could not export HTML map: {exc}")

m